<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Exercise: Add a Third Tool </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

## Purpose
Practice extending an agent with a new tool and improving tool selection behavior.

---

## Learning Goals
- Design a new tool
- Integrate it into an agent
- Improve decision-making logic
- Track which tools are used

## Scenario

Your AI assistant is improving.

Previously, it could:
- Retrieve data
- Calculate differences

Now, users want **more insights**.

---

## New Requirement

Users now ask:

"What is the percentage growth in revenue?"

---

## Problem

Your current agent:
- Can compute difference ✅
- Cannot compute percentage ❌

---

You must add a **new tool** and update the agent.

## Requirements

### You must:

1. Add a **third tool**
2. Update agent to use it when needed
3. Run multiple queries
4. Track which tool was used

---

### Required Tools
- retrieve_docs
- calculator
- YOUR NEW TOOL

---

### Output Must Include
- Reasoning trace
- Tool usage log

---

## Rules
- Do NOT hardcode answers
- Tool selection must depend on query

In [ ]:
import ast
import operator as op
from dataclasses import dataclass
from typing import Callable, Dict, Any

# Safe eval (same as demo)
_operators = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
}

def safe_eval(expr: str) -> float:
    node = ast.parse(expr, mode='eval')

    def _eval(n):
        if isinstance(n, ast.Expression):
            return _eval(n.body)
        if isinstance(n, ast.Constant):
            return n.value
        if isinstance(n, ast.BinOp):
            return _operators[type(n.op)](_eval(n.left), _eval(n.right))
        raise ValueError("Unsupported")

    return _eval(node)

In [ ]:
@dataclass
class Tool:
    name: str
    func: Callable[[str], Dict[str, Any]]
    description: str = ""
    returns_final: bool = False


class ToolRegistry:
    def __init__(self):
        self._tools = {}

    def register(self, tool: Tool):
        self._tools[tool.name] = tool

    def get(self, name: str):
        return self._tools[name]


registry = ToolRegistry()

Add a new tool: percentage_growth

Then update tool selection logic to use it correctly.

In [ ]:
# -------------------------------
# Tools (DO NOT MODIFY)
# -------------------------------

def retrieve_tool(query: str):
    return {
        "output": {"revenue_2023": 100, "revenue_2022": 80},
        "tool": "retrieve",
        "final": False
    }


def calculator_tool(query: str):
    try:
        # Safely convert string to dictionary
        if isinstance(query, str):
            data = ast.literal_eval(query)
        else:
            data = query

        # Extract values
        rev_2023 = data["revenue_2023"]
        rev_2022 = data["revenue_2022"]

        # Compute growth
        growth = rev_2023 - rev_2022

        return {"output": growth, "tool": "calculator", "final": False}

    except Exception as e:
        return {"output": f"error: {e}", "tool": "calculator", "final": False}


def answer_tool(query: str):
    return {"output": f"Final answer: {query}", "tool": "answer", "final": True}


registry.register(Tool("retrieve", retrieve_tool))
registry.register(Tool("calculator", calculator_tool))
registry.register(Tool("answer", answer_tool, returns_final=True))

In [ ]:
# -------------------------------
# TODO: Add Third Tool
# -------------------------------

import ast

def percentage_tool(query: str):
    try:
        # If it's a string, try converting to dict
        if isinstance(query, str):
            try:
                data = ast.literal_eval(query)
            except:
                # If not a dict string → data not ready yet
                return {
                    "output": "error: data not available yet",
                    "tool": "percentage",
                    "final": False
                }
        else:
            data = query

        # Now safely compute
        new = data["revenue_2023"]
        old = data["revenue_2022"]

        return {
            "output": ((new - old) / old) * 100,
            "tool": "percentage",
            "final": False
        }

    except Exception as e:
        return {
            "output": f"error: {e}",
            "tool": "percentage",
            "final": False
        }


registry.register(Tool("percentage_growth", percentage_tool))

In [ ]:
def choose_tool(state):
    # Step 1: if no data yet, always retrieve
    if state["data"] is None:
        return "retrieve", "need to retrieve data first"

    # Step 2: data exists, decide which computation tool
    if "percentage" in state["query"].lower():
        return "percentage_growth", "compute percentage growth"

    else:
        return "calculator", "compute revenue difference"

In [ ]:
def run_agent(query, max_steps=5):
    state = {"query": query, "data": None}

    for step in range(max_steps):
        print(f"\nStep {step+1}")

        tool_name, reason = choose_tool(state)
        print("Tool:", tool_name, "| Reason:", reason)

        tool = registry.get(tool_name)

        # Pass the correct input to tool
        input_data = state["data"] if state["data"] is not None else state["query"]

        result = tool.func(str(input_data))
        print("Output:", result["output"])

        # If the tool signals final, stop
        if result["final"]:
            return result

        # Store results safely
        if tool_name == "retrieve":
            state["data"] = result["output"]
        elif tool_name in ["calculator", "percentage_growth"]:
            state["data"] = result["output"]
            # after computation, go to answer
            answer = answer_tool(state["data"])
            print("Output:", answer["output"])
            return answer

    return {"output": "Max steps reached"}

In [ ]:
queries = [
    "Find revenue growth",
    "Find percentage growth",
    "Calculate difference"
]

for q in queries:
    print("\n=====", q)
    print(run_agent(q))


===== Find revenue growth

Step 1
Tool: retrieve | Reason: need to retrieve data first
Output: {'revenue_2023': 100, 'revenue_2022': 80}

Step 2
Tool: calculator | Reason: compute revenue difference
Output: 20
Output: Final answer: 20
{'output': 'Final answer: 20', 'tool': 'answer', 'final': True}

===== Find percentage growth

Step 1
Tool: retrieve | Reason: need to retrieve data first
Output: {'revenue_2023': 100, 'revenue_2022': 80}

Step 2
Tool: percentage_growth | Reason: compute percentage growth
Output: 25.0
Output: Final answer: 25.0
{'output': 'Final answer: 25.0', 'tool': 'answer', 'final': True}

===== Calculate difference

Step 1
Tool: retrieve | Reason: need to retrieve data first
Output: {'revenue_2023': 100, 'revenue_2022': 80}

Step 2
Tool: calculator | Reason: compute revenue difference
Output: 20
Output: Final answer: 20
{'output': 'Final answer: 20', 'tool': 'answer', 'final': True}


## Expected Behavior

Your agent should:

Step 1:
- Decide to retrieve data

Step 2:
- Decide to calculate growth

Step 3:
- Return final answer

---

## Expected Trace Format

===== Step 1 =====  
Decision: retrieve_docs  
Observation: {...}

===== Step 2 =====  
Decision: calculator  
Observation: 20

===== Step 3 =====  
Decision: final_answer  

Final Output: Revenue growth is 20

## Reflection Questions

1. What makes your system an **agent** instead of a pipeline?

2. Where is the **decision-making logic** implemented?

3. What would happen if:
   - The tool returned incorrect data?
   - The calculator failed?

4. How would you improve:
   - Tool selection?
   - Flexibility of the agent?

5. What limitations does your current agent have?